# 7.2 数据转换

## 7.2.1 删除重复数据

In [2]:
import pandas as pd
import numpy as np
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                     "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


In [3]:
# 返回bool型Series表示各行的列上的值在前面的行中出现过
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

In [4]:
data.drop_duplicates()  # 把上面的Series里的True的行删除掉

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


In [5]:
# 也可以指定部分列判断是否重复
data['v1'] = range(7)
data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [6]:
data.drop_duplicates(subset=['k1'])

,k1,k2,v1
0,one,1,0
1,two,1,1


In [7]:
# drop_duplicates默认保留的是第一个出现的值的组合 传入keep='last'则保留最后一个
data.drop_duplicates(['k1','k2'],keep='last')

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


## 7.2.2 利用函数或映射进行数据转换

In [8]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                              "pastrami", "corned beef", "bacon",
                              "pastrami", "honey ham", "nova lox"],
                     "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


In [9]:
meat_to_animal = {
  "bacon": "pig",
  "pulled pork": "pig",
  "pastrami": "cow",
  "corned beef": "cow",
  "honey ham": "pig",
  "nova lox": "salmon"
}

# 字典映射

data['animal'] = data['food'].map(meat_to_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


In [10]:
def get_animal(x):
    return meat_to_animal[x]
data['food'].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: str

## 7.2.3 替换值

In [11]:
data = pd.Series([1,-999,2,-99,-1000,3])
data

0       1
1    -999
2       2
3     -99
4   -1000
5       3
dtype: int64

In [12]:
data.replace(-999,np.nan)   # 不会修改原始值

0       1.0
1       NaN
2       2.0
3     -99.0
4   -1000.0
5       3.0
dtype: float64

In [13]:
data.replace([-999,-1000],np.nan)

0     1.0
1     NaN
2     2.0
3   -99.0
4     NaN
5     3.0
dtype: float64

In [14]:
data.replace([-999,-1000],[np.nan,0])

0     1.0
1     NaN
2     2.0
3   -99.0
4     0.0
5     3.0
dtype: float64

In [15]:
data.replace({-999:np.nan,-1000:1})

0     1.0
1     NaN
2     2.0
3   -99.0
4     1.0
5     3.0
dtype: float64

In [16]:
# 字符串的元素级替换
data = pd.Series(['apple', 'banana', 'apple pie', 'pineapple'])
result = data.str.replace('apple', 'orange')
print(result)

0        orange
1        banana
2    orange pie
3    pineorange
dtype: str


## 7.2.4 重命名轴索引

In [17]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


In [18]:
def transform(x):
    return x[:4].upper()
data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='str')

In [19]:
data.index = data.index.map(transform)
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


In [20]:
# rename
df = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
print(df)
df.rename(columns={'A': '甲', 'B': '乙'}, inplace=True)
print(df)

   A  B
0  1  3
1  2  4
   甲  乙
0  1  3
1  2  4


In [21]:
data.rename(index=str.title,columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


## 7.2.5 离散化和分箱

In [22]:
ages = [20,22,25,27,21,23,37,31,61,45,41,32]
bins = [18,25,35,60,100]
age_categories = pd.cut(ages,bins)
age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

In [23]:
age_categories.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [24]:
age_categories.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [25]:
age_categories.categories[0]

Interval(18, 25, closed='right')

In [26]:
age_categories.value_counts()

(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

In [27]:
# 左边是闭的可以通过right=False进行修改
pd.cut(ages,bins,right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

In [28]:
group_names = ['Youth','YoungAdult','MiddleAged','Senior']
pd.cut(ages,bins,labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, str): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

In [29]:
# 如果向pd.cut传入的不是确切的分箱边界,而是分箱的数据,则会根据数据的最小值和最大值计算得到等长的箱
data = np.random.uniform(size=20)
pd.cut(data,4,precision=2)  # precision=2表示限定小数点之后只有两位

[(0.27, 0.51], (0.035, 0.27], (0.51, 0.75], (0.51, 0.75], (0.51, 0.75], ..., (0.035, 0.27], (0.035, 0.27], (0.035, 0.27], (0.035, 0.27], (0.27, 0.51]]
Length: 20
Categories (4, interval[float64, right]): [(0.035, 0.27] < (0.27, 0.51] < (0.51, 0.75] < (0.75, 0.98]]

In [30]:
# pd.qcut使用样本分位数因此可以得到大小基本相等的分箱
data = np.random.randn(1000)
quartiles = pd.qcut(data,4,precision=2)
quartiles

[(-0.054, 0.69], (-0.67, -0.054], (-0.67, -0.054], (-3.3299999999999996, -0.67], (0.69, 3.84], ..., (-0.67, -0.054], (-3.3299999999999996, -0.67], (0.69, 3.84], (-0.054, 0.69], (-3.3299999999999996, -0.67]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.3299999999999996, -0.67] < (-0.67, -0.054] < (-0.054, 0.69] < (0.69, 3.84]]

In [31]:
pd.Series(quartiles).value_counts(ascending=True)  # 将分类数据转成Series后才能在value_counts方法中传参 当热这里是为了表明qcut分类在基数上是均匀的

(-3.3299999999999996, -0.67]    250
(-0.67, -0.054]                 250
(-0.054, 0.69]                  250
(0.69, 3.84]                    250
Name: count, dtype: int64

In [32]:
# 可以传递自定义的分位数(0到1之间的数值,包含端点)
pd.qcut(data,[0,0.1,0.5,0.9,1]).value_counts()

(-3.316, -1.297]     100
(-1.297, -0.0537]    400
(-0.0537, 1.252]     400
(1.252, 3.842]       100
Name: count, dtype: int64

## 7.2.6 检测和过滤异常值

In [33]:
data = pd.DataFrame(np.random.randn(1000,4))
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.034284,-0.029256,0.042931,-0.011815
std,0.992103,0.975622,0.959698,1.028351
min,-3.501031,-2.841793,-3.066426,-3.571458
25%,-0.630261,-0.691582,-0.570697,-0.714994
50%,0.023425,-0.008102,0.043301,0.008844
75%,0.672729,0.604902,0.711550,0.671586
max,2.942446,3.263388,3.084227,3.044688


In [34]:
col = data[2]
col[col.abs()>3]

27     3.010052
435    3.084227
557    3.013279
810   -3.066426
Name: 2, dtype: float64

In [35]:
# 要选出全部含有绝对值大于3的行 可以在bool型DataFrame中使用any方法
data[(data.abs()>3).any(axis=1)]

,0,1,2,3
27,-0.023056,-0.617233,3.010052,-0.480174
106,0.110235,3.263388,0.011876,0.343709
146,-0.850114,-0.317271,1.534091,3.000513
202,0.592395,0.739937,-0.113961,-3.057210
298,-0.151373,0.471177,0.777710,-3.571458
435,1.363333,-1.339803,3.084227,-0.029037
538,-3.501031,-0.138421,1.386342,0.129866
557,1.761760,0.285710,3.013279,-0.829273
597,-0.553632,0.815148,1.068327,3.044688
608,-0.963401,-1.379703,1.438877,-3.204880


In [36]:
# 将所有值限制在[-3,3]
data[data.abs()>3] = np.sign(data)*3
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.034888,-0.029519,0.042890,-0.011017
std,0.990121,0.974767,0.959150,1.025580
min,-3.000000,-2.841793,-3.000000,-3.000000
25%,-0.630261,-0.691582,-0.570697,-0.714994
50%,0.023425,-0.008102,0.043301,0.008844
75%,0.672729,0.604902,0.711550,0.671586
max,2.942446,3.000000,3.000000,3.000000


## 7.2.7 置换和随机采样

In [37]:
sampler = np.random.permutation(5)
sampler

array([1, 2, 3, 4, 0], dtype=int32)

In [38]:
df = pd.DataFrame(np.arange(35).reshape(5,7))
df

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


In [39]:
df.iloc[sampler]

,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34
0,0,1,2,3,4,5,6


In [40]:
df.take(sampler)

,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34
0,0,1,2,3,4,5,6


In [41]:
column_sampler = np.random.permutation(7)
column_sampler

array([3, 1, 6, 5, 4, 2, 0], dtype=int32)

In [42]:
df.take(column_sampler,axis=1)

,3,1,6,5,4,2,0
0,3,1,6,5,4,2,0
1,10,8,13,12,11,9,7
2,17,15,20,19,18,16,14
3,24,22,27,26,25,23,21
4,31,29,34,33,32,30,28


In [43]:
df.sample(3)  # 无重复

,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
4,28,29,30,31,32,33,34


In [44]:
choices = pd.Series([5,7,-1,6,4])
choices.sample(n=10,replace=True)  # 可重复

1    7
0    5
0    5
3    6
0    5
1    7
1    7
0    5
4    4
3    6
dtype: int64

## 7.2.8 计算指标/虚拟变量

In [45]:
df = pd.DataFrame({'key':['b','b','a','c','a','b'],'data1':range(6)})
df

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [47]:
# get_dummies 分类变量转哑变量,常用于机器学习特征处理
pd.get_dummies(df['key']).astype(int)

,a,b,c
0,0,1,0
1,0,1,0
2,1,0,0
3,0,0,1
4,1,0,0
5,0,1,0


In [49]:
dummies = pd.get_dummies(df['key'],prefix='key')
df_with_dummy = df[['data1']].join(dummies)  # 只有DataFrame才能调用join
df_with_dummy

,data1,key_a,key_b,key_c
0,0,False,True,False
1,1,False,True,False
2,2,True,False,False
3,3,False,False,True
4,4,True,False,False
5,5,False,True,False


In [51]:
mnames = ['movie_id','title','genres']
movies = pd.read_table('../datasets/movielens/movies.dat', sep='::', names=mnames, engine='python', header=None)
movies[:10]

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children's
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [53]:
dummies = movies['genres'].str.get_dummies('|')
dummies.iloc[:10,:6]

,Action,Adventure,Animation,Children's,Comedy,Crime
0,0,0,1,1,1,0
1,0,1,0,1,0,0
2,0,0,0,0,1,0
3,0,0,0,0,1,0
4,0,0,0,0,1,0
5,1,0,0,0,0,1
6,0,0,0,0,1,0
7,0,1,0,1,0,0
8,1,0,0,0,0,0
9,1,1,0,0,0,0


In [54]:
movies_windic = movies.join(dummies.add_prefix('Genre_'))
movies_windic.iloc[0]

movie_id                                       1
title                           Toy Story (1995)
genres               Animation|Children's|Comedy
Genre_Action                                   0
Genre_Adventure                                0
Genre_Animation                                1
Genre_Children's                               1
Genre_Comedy                                   1
Genre_Crime                                    0
Genre_Documentary                              0
Genre_Drama                                    0
Genre_Fantasy                                  0
Genre_Film-Noir                                0
Genre_Horror                                   0
Genre_Musical                                  0
Genre_Mystery                                  0
Genre_Romance                                  0
Genre_Sci-Fi                                   0
Genre_Thriller                                 0
Genre_War                                      0
Genre_Western       

In [56]:
np.random.seed(12345)
values = np.random.uniform(size=10)
values

array([0.92961609, 0.31637555, 0.18391881, 0.20456028, 0.56772503,
       0.5955447 , 0.96451452, 0.6531771 , 0.74890664, 0.65356987])

In [57]:
bins = [0,0.2,0.4,0.6,0.8,1]
pd.get_dummies(pd.cut(values,bins))

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,True
1,False,True,False,False,False
2,True,False,False,False,False
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,True,False,False
6,False,False,False,False,True
7,False,False,False,True,False
8,False,False,False,True,False
9,False,False,False,True,False


# End